# 第 21 节: RL 调试方法论

## 学习目标
1. 掌握 RL 训练中的常见故障模式
2. 学会系统性地诊断问题
3. 能根据训练曲线判断可能的原因
4. 完成一个故意写错的 PPO 调试练习 (无标签版本)

## 1. RL 调试的特殊挑战

RL 调试比监督学习更难，因为:
- **非平稳性**: 数据分布随策略改变而变化
- **延迟反馈**: 奖励可能在很多步后才出现
- **多组件交互**: Actor × Critic × Buffer × Env
- **随机性大**: 同一配置不同种子结果可能完全不同
- **没有正确答案**: 没有标准测试集

## 2. 调试检查清单

### 第一步: 环境检查
- Observation 的 shape 和 range 是否正确?
- Reward scale 是否合理? (推荐 [-1, 1] 或 [0, 1])
- Terminated vs Truncated 是否正确处理?
- 随机策略的 episode return 基线是多少?

### 第二步: 网络检查
- 网络输入/输出维度是否匹配?
- 初始化是否合理? (推荐正交初始化)
- 梯度是否正常流动? (grad_norm)

### 第三步: 算法检查
- Advantage 是否 detach?
- Old log prob 是否正确保存?
- Return 和 Advantage 是否混淆?
- Bootstrap 边界条件是否正确?

### 第四步: 训练过程
- Reward 是否上升?
- Loss 是否在合理范围?
- Entropy 是否下降太快?
- 有没有 NaN?

## 3. 常见故障模式及诊断

### 故障 1: Reward 不上升
可能原因: 学习率太小/太大、网络太浅、训练不够久
诊断: 先跑随机策略看 return 范围，再逐步调大 lr

### 故障 2: Loss 震荡剧烈
可能原因: batch size 太小、lr 太大、return 未标准化
诊断: 增大 batch size、降低 lr、添加 return normalization

### 故障 3: Entropy 过快下降
可能原因: 策略过早收敛到确定性策略
诊断: 增大 entropy coefficient、减小 lr

### 故障 4: NaN 出现
可能原因: 梯度爆炸、log(0)、除零
诊断: 添加 grad clipping、检查 log_prob 计算、添加 eps

### 故障 5: Critic loss 很大且不下降
可能原因: return scale 太大、网络不够强
诊断: reward normalization、更深的 Critic

### 故障 6: Episode return 上升但 evaluation return 不上升
可能原因: 训练时用了 exploration，eval 时没开
诊断: 确保 eval 用 deterministic 模式

## 4. 诊断练习: 找出下面 PPO 代码中的所有问题

下面是一个看起来正常的 PPO 实现。请仔细阅读代码，找出其中的所有问题。

### 问题分类

为了帮助你系统性地思考，我们将问题分为三个等级:

**Tier 1 — 确定会造成错误的 Bug (3 个)**
这些错误会导致训练完全失败或产生完全错误的结果。必须修复。

- (a) `old_log_probs` 在 ratio 计算前被覆盖，导致 ratio ≈ 1，PPO 退化为普通的 policy gradient
- (b) ratio 方向错误: 对于 advantage < 0 的情况，clipping 的方向反了，导致策略更新不稳定
- (c) bootstrap 值设置错误: truncated (截断) 时 bootstrap 值应为 V(s')，但代码提供了 0.0

**Tier 2 — 高风险实现问题 (2 个)**
这些问题不一定会导致 NaN 或 crash，但会严重影响训练的正确性和监控可靠性。

- (d) advantages 在 policy loss 中未 detach，导致梯度通过 advantages 反向传播到 critic 网络
- (e) KL 散度在相同分布上计算，监控指标永远接近 0，无法反映真实的策略变化

**Tier 3 — 可选稳定化技巧 (3 个)**
这些不是错误，但添加后可以显著提升训练稳定性。好的 PPO 实现通常会包含它们。

- (f) advantage normalization (标准化 advantages)
- (g) entropy bonus (熵奖励，鼓励探索)
- (h) gradient clipping (梯度裁剪)

> **任务**: 先不看下方的答案，尝试找出三个等级中的所有问题。对每个问题，解释为什么它是个问题以及如何修复。

In [ ]:
# PPO 调试练习 — 请找出代码中的所有问题
# 提示：问题分布在三个等级中，请参考上面的说明

import torch
import torch.nn as nn
from torch.distributions import Categorical


class DiagnosticPPO:
    """
    一个有问题的 PPO 实现。

    这个实现看起来可以运行，但包含多个类型的问题。
    请仔细阅读并找出所有问题 (不限于代码中的标记问题)。
    """

    def __init__(self,
                 actor: nn.Module,
                 critic: nn.Module,
                 optimizer: torch.optim.Optimizer,
                 gamma: float = 0.99,
                 lam: float = 0.95,
                 clip_epsilon: float = 0.2):
        self.actor = actor
        self.critic = critic
        self.optimizer = optimizer
        self.gamma = gamma
        self.lam = lam
        self.clip_epsilon = clip_epsilon

    def compute_gae(self, rewards, dones, values, next_value):
        """
        计算 Generalized Advantage Estimation (GAE)。

        Args:
            rewards: 奖励序列 [T]
            dones: 终止标志 [T] (1 = 终止或截断, 0 = 未终止)
            values: 价值预测 [T]
            next_value: 最后一个状态的 V(s_{T+1})
        """
        advantages = []
        gae = 0.0
        for t in reversed(range(len(rewards))):
            if dones[t]:
                next_val = 0.0
            else:
                next_val = values[t + 1] if t < len(rewards) - 1 else next_value
            delta = rewards[t] + self.gamma * next_val - values[t]
            gae = delta + self.gamma * self.lam * (1 - dones[t]) * gae
            advantages.insert(0, gae)
        return torch.stack(advantages)

    def update(self, states, actions, old_log_probs, rewards, dones, next_state):
        """
        执行一次 PPO 更新 (多 epoch 调用)。

        Args:
            states: 状态 [T, obs_dim]
            actions: 动作 [T]
            old_log_probs: 数据收集时保存的 log_prob [T]
            rewards: 奖励 [T]
            dones: 终止标志 [T]
            next_state: 最后一个状态 [obs_dim]
        """
        # --- Critic 前向 ---
        values = self.critic(states).squeeze(-1)
        with torch.no_grad():
            next_value = self.critic(next_state.unsqueeze(0)).squeeze(-1)

        # --- 计算 Advantage & Return ---
        advantages = self.compute_gae(rewards, dones, values, next_value)
        returns = advantages + values.detach()

        # --- Actor 前向 ---
        logits = self.actor(states)
        dist = Categorical(logits=logits)

        # 计算新策略的 log_prob
        old_log_probs = dist.log_prob(actions)
        new_log_probs = dist.log_prob(actions)

        # PPO ratio
        ratio = torch.exp(new_log_probs - old_log_probs)

        # --- Policy Loss (PPO clipped objective) ---
        surr1 = ratio * advantages
        surr2 = (torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon)
                 * advantages)
        policy_loss = -torch.max(surr1, surr2).mean()

        # --- Value Loss (MSE) ---
        value_loss = (values - returns).pow(2).mean()

        # --- Total Loss & Update ---
        total_loss = policy_loss + value_loss

        self.optimizer.zero_grad()
        total_loss.backward()
        self.optimizer.step()

        return {
            'policy_loss': policy_loss.item(),
            'value_loss': value_loss.item(),
            'ratio_mean': ratio.mean().item(),
        }


# ===== 训练循环中的问题 (参考) =====

def training_loop_with_issues(ppo, states, actions, old_log_probs,
                               rewards, dones, next_state):
    """训练循环，包含 KL 监控问题。"""
    num_epochs = 10
    for epoch in range(num_epochs):
        # PPO 更新
        info = ppo.update(states, actions, old_log_probs,
                          rewards, dones, next_state)

        # KL 监控 — 请检查这里是否有问题
        with torch.no_grad():
            current_logits = ppo.actor(states)
            # 计算 KL 用于监控策略变化幅度
            approx_kl = ((current_logits - current_logits) ** 2).mean()
        print(f"Epoch {epoch}, KL: {approx_kl.item():.6f}")


print("代码加载完成。请仔细阅读上面的 DiagnosticPPO 类。")
print("尝试在不看答案的情况下找出所有 Tier 1-3 的问题!")

## 5. 正确的调试策略

1. **从小处开始**: 先用最简单的环境 (CartPole) 验证
2. **一次改一个**: 每次只改一个超参数
3. **看多个指标**: 不只是 reward, 还有 loss, entropy, KL, clip fraction
4. **多跑几个种子**: 至少 3-5 个随机种子
5. **对比已知正确实现**: 用成熟库做 baseline
6. **可视化一切**: 不要只看数字

## 6. 练习

1. 修复上面的 `DiagnosticPPO`，解决所有 Tier 1 和 Tier 2 的问题
2. 添加 Tier 3 中的三个可选稳定化技巧
3. 在你的 PPO 实现中故意引入单个 error，看是否能从训练曲线中识别出它
4. 收集至少 5 个不同种子的训练曲线，计算均值和标准差

---
下一节: 22_experiment_design.ipynb

<details>
<summary><b>点击查看完整答案</b></summary>

---

### Tier 1 — 确定会造成错误的 Bug (3 个)

#### (a) old_log_probs 在 ratio 计算前被覆盖

**问题位置**: `update` 方法中:
```python
old_log_probs = dist.log_prob(actions)   # ← 覆盖了参数!
new_log_probs = dist.log_prob(actions)
ratio = torch.exp(new_log_probs - old_log_probs)  # ratio ≈ 1
```

参数 `old_log_probs` 来自数据收集阶段，保存的是旧策略下的 log probability。
但代码中用当前策略的 `dist.log_prob(actions)` 将其覆盖了，导致 `new_log_probs` 和
`old_log_probs` 完全相等，ratio ≈ 1 对每个样本都成立。PPO 的 clipped objective
退化为普通的 policy gradient（而且由于 `max` 的问题，方向也错了）。

**修复方案**: 不要覆盖参数。保留传入的 `old_log_probs`:
```python
new_log_probs = dist.log_prob(actions)
ratio = torch.exp(new_log_probs - old_log_probs)  # old_log_probs 来自参数
```

---

#### (b) ratio 方向错误 (使用 max 而非 min)

**问题位置**:
```python
policy_loss = -torch.max(surr1, surr2).mean()  # ← 应该用 min
```

PPO 的标准目标是 `L = E[min(r*A, clip(r)*A)]`。使用 `max` 会**逆转 clipping 的方向**:

| 场景 | min (正确) | max (错误) |
|------|-----------|-----------|
| A > 0, r > 1+ε | 取 clip(r)*A，限制增幅 | 取 r*A，**不限制增幅** |
| A < 0, r < 1-ε | 取 clip(r)*A，限制降幅 | 取 r*A，**继续降低概率** |
| A < 0, r > 1+ε | 取 r*A，完整惩罚 | 取 clip(r)*A，**削弱惩罚** |

特别是对 A < 0 且 r < 1-ε 的情况（策略已经大幅降低了一个坏动作的概率），
`max` 会继续施加梯度降低概率，导致策略更新不稳定。

**修复方案**:
```python
policy_loss = -torch.min(surr1, surr2).mean()
```

---

#### (c) Truncated 时 bootstrap 值错误

**问题位置**: `compute_gae` 方法中:
```python
if dones[t]:
    next_val = 0.0  # ← 截断 (truncated) 时不应该用 0
```

RL 中有两种 "episode 结束":
- **Terminated**: 环境自然结束（如到达目标、撞墙）→ bootstrap = 0 ✓
- **Truncated**: 达到最大步数限制 → bootstrap = V(s') 估计未来回报

代码使用单一的 `dones` 标志，对 truncated 的 episode 也使用 `next_val = 0.0`，
导致 bootstrap 被低估。这会使价值函数学习到错误的目标，低估长期回报。

**修复方案**: 分离 terminated 和 truncated 标志:
```python
if terminated[t]:
    next_val = 0.0  # 自然终止
elif truncated[t]:
    next_val = values[t + 1]  # 截断，需要 bootstrap
else:
    next_val = values[t + 1]  # 未终止
```
或者等价地:
```python
if terminated[t]:
    next_val = 0.0
else:
    next_val = values[t + 1] if t < len(rewards) - 1 else next_value
```

---

### Tier 2 — 高风险实现问题 (2 个)

#### (d) Advantages 未 detach 导致梯度流入 Critic

**问题位置**:
```python
advantages = self.compute_gae(rewards, dones, values, next_value)
# ...
policy_loss = -torch.max(surr1, surr2).mean()  # advantages 未 detach
```

`advantages` 通过 `compute_gae` 依赖于 `values` (self.critic 的输出)，
因此 `advantages` 包含 critic 的计算图。当 `policy_loss` 反向传播时，
梯度会通过 `advantages` 流入 critic 网络，错误地更新了 critic 参数。

这与 value loss 的梯度叠加，导致 critic 的更新方向不正确。

**修复方案**:
```python
advantages = advantages.detach()  # 切断与 critic 的连接
# 或在 policy loss 中使用 detached advantages:
policy_loss = -torch.max(surr1, surr2).mean()  # surr1/2 使用 detached advantages
```

---

#### (e) KL 散度在相同分布上计算

**问题位置**: `training_loop_with_issues` 函数中:
```python
current_logits = ppo.actor(states)
approx_kl = ((current_logits - current_logits) ** 2).mean()  # Always 0!
```

这里 `current_logits - current_logits` 恒等于 0，所以 KL 监控永远返回 0。
即使修复了 (a) 中的覆盖问题，KL 监控本身仍然是错误的。这会给训练者一种
"策略没有变化"的假象，掩盖策略崩溃或更新过大的问题。

此外，即使计算方式正确，代码在更新前也没有保存旧策略的分布，无法计算
真正的 `KL(old || new)`。

**修复方案**:
```python
# 在 PPO 更新前保存旧策略 logits
with torch.no_grad():
    old_logits = ppo.actor(states).detach().clone()

# 执行 PPO 更新 ...

# 更新后计算 KL
with torch.no_grad():
    new_logits = ppo.actor(states)
    # 真正的 KL 散度 (使用分布，非近似)
    old_dist = Categorical(logits=old_logits)
    new_dist = Categorical(logits=new_logits)
    kl = kl_divergence(old_dist, new_dist).mean().item()
```

---

### Tier 3 — 可选稳定化技巧 (3 个)

#### (f) Advantage Normalization

在 policy loss 之前，对 advantages 进行标准化，能稳定训练:

```python
# 在 update 方法中，使用 advantages 前添加:
advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
```

**作用**: 将 advantage 的均值归一化到 0，方差到 1，使得策略更新的幅度
不受 reward scale 的影响。

---

#### (g) Entropy Bonus

在总损失中加入策略熵的负项，鼓励探索，防止策略过早坍缩:

```python
# 在 actor 前向后、total_loss 计算前:
entropy = dist.entropy().mean()
total_loss = policy_loss + value_loss - ent_coef * entropy
```

其中 `ent_coef` 是一个超参数（通常 0.01），控制熵正则化的强度。
熵下降过快通常意味着策略过早收敛到确定性策略。

---

#### (h) Gradient Clipping

在 `backward()` 后、`optimizer.step()` 前添加梯度裁剪:

```python
total_loss.backward()
torch.nn.utils.clip_grad_norm_(self.actor.parameters(), max_norm=0.5)
torch.nn.utils.clip_grad_norm_(self.critic.parameters(), max_norm=0.5)
self.optimizer.step()
```

**作用**: 防止梯度爆炸导致的 NaN 或策略突变。特别是在 PPO 中，
多 epoch 更新时梯度容易累积变大，gradient clipping 是一道重要的安全网。

---

### 修正后的 update 方法 (完整参考)

```python
def update(self, states, actions, old_log_probs, rewards,
           terminated, truncated, next_state):
    # --- Critic ---
    values = self.critic(states).squeeze(-1)
    with torch.no_grad():
        next_value = self.critic(next_state.unsqueeze(0)).squeeze(-1)

    # --- Advantage (fixed: separate terminated vs truncated) ---
    advantages = self.compute_gae_fixed(
        rewards, terminated, truncated, values, next_value)
    returns = advantages + values.detach()

    # --- Actor ---
    logits = self.actor(states)
    dist = Categorical(logits=logits)
    new_log_probs = dist.log_prob(actions)  # 不覆盖参数
    ratio = torch.exp(new_log_probs - old_log_probs)   # old_log_probs 来自参数

    # --- Policy Loss (fixed: min, detached advantages) ---
    adv = advantages.detach()
    surr1 = ratio * adv
    surr2 = torch.clamp(ratio, 1 - self.clip_epsilon,
                                1 + self.clip_epsilon) * adv
    policy_loss = -torch.min(surr1, surr2).mean()

    # --- Value Loss ---
    value_loss = (values - returns).pow(2).mean()

    # --- Entropy Bonus ---
    entropy = dist.entropy().mean()
    total_loss = policy_loss + value_loss - 0.01 * entropy

    # --- Optimize (with gradient clipping) ---
    self.optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(self.actor.parameters(), 0.5)
    torch.nn.utils.clip_grad_norm_(self.critic.parameters(), 0.5)
    self.optimizer.step()

    return {
        'policy_loss': policy_loss.item(),
        'value_loss': value_loss.item(),
        'entropy': entropy.item(),
        'ratio_mean': ratio.mean().item(),
    }
```

</details>